# 📗 DPO 선호 정렬: 노무 상담 모델이 범위 밖 질문을 거절하도록 학습하기

#### **선호 정렬(preference alignment)**

> 같은 질문의 두 답 가운데 **더 낫다고 정한 답**을 모델이 **더 자주 내도록** 학습하는 일입니다. 어느 답이 나은지는 데이터를 만드는 쪽이 정합니다.

| | SFT | 선호 정렬 |
|---|---|---|
| 보여 주는 답 | 정답 답변 하나 | 더 나은 답과 더 나쁜 답 |
| '이렇게 답하면 안 된다'는 신호 | 없음 | 있음(더 나쁜 답의 확률을 낮출 수 있음) |

- **예:** 노무 상담 모델이 상담 범위 밖의 질문에도 노동법을 끌어와 답을 지어 냅니다. 거절 문장을 더 나은 답, 지어 낸 답을 더 나쁜 답으로 묶어 학습하면 이런 질문에 **거절하는 쪽으로** 바뀝니다.
- **한계:** 선호 쌍에 담긴 **답 사이의 우열만 가르칩니다.** 어느 답에도 없는 지식은 넣지 못합니다.

<img src="images/01_sft_then_dpo.png" width="1050" alt="SFT와 DPO의 차이: SFT는 범위 밖 질문에 거절 문장을 따라 쓰게 할 뿐이고, DPO는 지어 낸 답(rejected)을 함께 보여 주어 그 확률을 직접 낮춥니다.">

이 교안은 선호 정렬 방법 가운데 DPO(1절)로 SFT 단원의 노무 상담 모델을 정렬합니다. 모델이 실제로 지어 낸 답으로 선호 쌍을 만들고, LoRA 어댑터를 beta 0.1과 0.5로 3에폭씩 학습해 에폭마다 저장합니다. 교안 02는 이 어댑터 6개를 채점해 정렬 모델을 고릅니다.

<img src="images/00_lesson01_goals.png" width="1050" alt="교안 01에서 배울 네 가지: 1 SFT 모델의 실패, 2 DPO 원리(참조 모델, 보상, beta), 3 선호 쌍 만들기, 4 DPO 학습과 beta 두 값 비교.">

**학습 목표** (그림의 번호와 같습니다)

1. **SFT 모델이 범위 밖 질문에 답을 지어 내는 이유와, DPO가 이를 고치는 방법을 설명합니다.**
2. **참조 모델, 보상, beta로 DPO 손실이 무엇을 줄이는지 설명합니다.**
3. **SFT 모델이 낸 답으로 범위 밖 쌍과 노무 질문 쌍을 만듭니다.**
4. **LoRA 어댑터로 DPO를 beta 두 값으로 학습하고, 학습 기록을 읽고 확인 질문 다섯 개의 답이 바뀌었는지 봅니다.**

## 1. 선호 쌍과 DPO

#### **선호 쌍(preference pair)**

> 같은 질문에 대한 **더 나은 답과 더 나쁜 답을 묶은** 학습 데이터 한 행입니다.

한 행은 세 열로 이루어집니다. 예는 4-3절에서 만드는 범위 밖 쌍 하나입니다(강사 실행 결과, `rejected`는 실행마다 다를 수 있습니다).

| 열 | 들어가는 것 | 예 |
|---|---|---|
| `prompt` | 시스템 메시지와 질문 | 노동조합을 새로 만들려면 어디에 무엇을 신고해야 하나요? |
| `chosen` | 더 나은 답 | 이 상담 챗봇이 다루는 노동법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요. |
| `rejected` | 더 나쁜 답 | 새 노동조합의 구성·설립 등에 관한 사항은 노동위원회에 신고해야 합니다. |

- **이 쌍이 가르치는 것:** 상담 범위 밖의 노동조합 질문에는 그럴듯한 답(`rejected`)보다 **거절 문장(`chosen`)을 더 자주 내라**는 것입니다.

#### **DPO(Direct Preference Optimization, 직접 선호 최적화)**

> 선호 쌍에서 참조 모델과 비교해 **`chosen` 쪽으로 기울도록**(`chosen`의 로그 확률은 `rejected`보다 더 올리거나 덜 내리도록) 모델을 학습하는 선호 정렬 방법입니다([Rafailov et al., 2023](https://arxiv.org/abs/2305.18290)).

- **정렬 모델:** 이 교안은 SFT 모델에 DPO로 학습한 LoRA 어댑터를 얹은 모델을 정렬 모델이라 부릅니다.

#### **SFT 모델의 실패**

> SFT 단원 평가에서 SFT 모델은 **범위 밖 질문 12개 중 8개에 답을 지어 냈습니다**(거절률 4/12).

- **SFT 모델:** SFT 단원에서 노무 상담 데이터로 파인튜닝한 Qwen3.5-0.8B입니다.
- **범위 밖 질문:** 다섯 법(근로기준법 등)으로 답할 수 없는 질문입니다.
- **예:** 운동 질문에도 "운동을 시작한 때부터 15분 이내에 …"처럼 노동법 문장으로 답했습니다.

## 2. DPO 원리

### 2-1. 참조 모델과 보상

#### **참조 모델(reference model)**

> 학습하지 않고 **고정해 두고**, 학습 중 모델과 **비교하는 데 쓰는 모델**입니다. 보통 정렬을 시작하는 모델을 쓰며, 이 교안은 SFT 모델입니다.

- **목적:** 학습 중 모델이 답마다 확률을 얼마나 바꿨는지를 이 모델과 비교해 계산합니다(아래 보상).
- **beta와의 관계:** DPO는 '사람이 선호하는 답일수록 높은 점수를 받게 하되, 참조 모델과 크게 달라지지는 말라'는 목표에서 나왔고, 그 목표를 풀면 아래 보상 식이 나옵니다. beta는 뒤쪽 조건(달라지는 데 주는 벌)의 무게입니다(2-3절).

#### **보상(reward)**

> 학습 중 모델이 답 하나의 확률을 참조 모델보다 **얼마나 올렸는지(양수), 내렸는지(음수)** 나타내는 점수입니다.

- **시작값:** 학습을 시작할 때는 두 모델이 같아 모든 답의 보상이 0입니다.
- **확률 대신 변화를 보는 이유:** 답의 확률은 토큰 확률을 모두 곱한 값이라 답마다 출발점(길이 등)이 크게 다릅니다. 같은 답을 참조 모델이 낸 확률과 견주면 이 출발점이 빠지고 **학습으로 바뀐 만큼만 남습니다.** 다만 바뀐 양은 토큰마다 더해지므로 긴 답일수록 커질 수 있습니다(4-4 길이 편향).

답 하나의 보상은 두 모델이 그 답에 준 로그 확률의 차이에 beta를 곱한 값입니다.

$\displaystyle r(x, y) = \beta \left( \log \pi_\theta(y \mid x) - \log \pi_{\text{ref}}(y \mid x) \right)$

- **$x$:** prompt(시스템 메시지와 질문)입니다.
- **$y$:** 답 하나입니다. 선호 쌍의 두 답은 아래처럼 씁니다.
  - **$y_w$:** `chosen`(더 나은 답)입니다. w는 이긴 답(winner)을 뜻합니다.
  - **$y_l$:** `rejected`(더 나쁜 답)입니다. l은 진 답(loser)을 뜻합니다.
- **$\pi_\theta(y \mid x)$:** 학습 중 모델이 $x$를 받고 답 $y$를 낼 확률입니다. $\pi$는 모델(정책, policy)을 뜻하는 기호이고, $\theta$는 학습으로 바뀌는 파라미터(이 교안은 LoRA 어댑터)입니다.
- **$\pi_{\text{ref}}(y \mid x)$:** 참조 모델이 같은 $x$를 받고 같은 답 $y$를 낼 확률입니다. 학습 내내 바뀌지 않습니다.
- **$\log$:** 자연로그입니다. 답의 확률은 토큰 확률을 모두 곱한 값이라, 로그를 씌우면 토큰마다의 로그 확률을 더한 값이 됩니다.
- **$\beta$:** 로그 확률의 차이에 곱하는 수로, `DPOConfig`의 `beta`입니다(2-3절).
- **예:** 학습 중 모델의 로그 확률이 참조 모델보다 2 높으면(확률로는 약 $e^2 \approx 7.4$배, $e$는 자연로그의 밑 약 2.718), beta 0.1에서 보상은 0.1 * 2 = 0.2입니다.

### 2-2. 손실

#### **margin**

> 한 쌍에서 **`chosen`의 보상에서 `rejected`의 보상을 뺀 값**, 곧 $r(x, y_w) - r(x, y_l)$입니다.

- **뜻:** 양수면 학습 중 모델이 참조 모델보다 `chosen` 쪽으로 기운 것이고, 음수면 `rejected` 쪽으로 기운 것입니다.
- **차이만 보는 이유:** 선호 쌍은 어느 답이 나은지만 알려 줄 뿐, 각 답이 몇 점인지는 알려 주지 않습니다.

<img src="images/03_dpo_principle.png" width="1050" alt="보상과 margin: chosen(더 나은 답, 예: 거절 문장)의 보상이 +0.2, rejected(더 나쁜 답, 예: 지어 낸 답)의 보상이 -0.3이면 margin은 0.5이고, margin이 클수록 손실이 작아집니다.">

#### **DPO 손실**

> **margin이 클수록 작아지는** 손실입니다. margin을 시그모이드에 넣어 0과 1 사이의 확률로 바꾸고(아래 $\sigma(\text{margin})$의 뜻), 그 확률에 $-\log$를 씌웁니다.

- **이진 교차 엔트로피와 같은 식:** 딥러닝 단원의 이진 교차 엔트로피에서 로짓 자리에 margin을, 정답 자리에 1(`chosen`이 낫다)을 넣은 식과 같습니다.

딥러닝 단원의 이진 교차 엔트로피는 다음 식입니다(이 교안은 $y$가 답을 뜻하므로 정답을 $c$로 씁니다).

$\displaystyle L_{\text{BCE}} = -\left[\, c \log p + (1 - c) \log (1 - p) \,\right], \quad p = \sigma(z)$

- **$c$:** 정답(1 또는 0)입니다.
- **$z$, $p$:** 로짓과, 로짓을 시그모이드 $\sigma$에 넣어 얻은 '정답이 1일 확률'입니다.
- **DPO로 바꾸기:** $z$에 margin, $c$에 1을 넣으면 뒤 항 $(1 - c) \log (1 - p)$가 0이 되어 $-\log \sigma(\text{margin})$만 남습니다. 이것이 아래 DPO 손실입니다.

$\displaystyle L_{\text{DPO}} = -\log \sigma\big( r(x, y_w) - r(x, y_l) \big)$

- **$r(x, y_w) - r(x, y_l)$:** margin(`chosen`의 보상 - `rejected`의 보상)입니다.
- **$\sigma$:** 시그모이드 함수 $\sigma(z) = 1 / (1 + e^{-z})$로, 값을 0과 1 사이로 바꿉니다.
- **$\sigma(\text{margin})$의 뜻:** 두 답의 보상으로 계산한 '`chosen`이 `rejected`보다 나을 확률'입니다(DPO 논문이 쓰는 브래들리-테리(Bradley-Terry) 선호 모델).
  - **주의:** 보상은 참조 모델 대비 변화이므로, 이 확률은 학습 중 모델이 참조 모델보다 어느 답 쪽으로 기울었는지를 나타냅니다. 모델이 실제로 `chosen`을 낼 확률이 아닙니다.
  - margin이 0이면 $\sigma(0) = 0.5$입니다. 참조 모델에 비해 어느 답 쪽으로도 기울지 않은 상태입니다(학습을 시작할 때).
  - margin이 양수면 0.5보다 크고, 클수록 1에 가까워집니다. `chosen` 쪽으로 기운 것입니다.
  - margin이 음수면 0.5보다 작습니다. `rejected` 쪽으로 기운 것입니다.
- **$-\log$:** $\sigma$ 값이 1에 가까울수록 손실이 0에 가까워지게 합니다.
- **학습 첫 DPO 손실:** 두 보상이 모두 0이라 $-\log \sigma(0) = \log 2 \approx 0.693$입니다.
- **학습기:** `DPOTrainer`는 쌍마다 이 손실을 계산해 배치 평균을 냅니다. `DPOConfig` 기본값 `loss_type=["sigmoid"]`가 이 식입니다.

<img src="images/08_dpo_loss_curve.png" width="1050" alt="DPO 손실 -log σ(margin)은 margin이 커질수록 0에 가까워집니다. 학습 시작(margin 0)의 손실은 0.693, 그림 03의 margin 0.5에서는 0.474입니다.">

**손실 곡선 읽기**
- margin이 음수인 쪽은 가파르고 큰 양수인 쪽은 평평합니다. 그래서 학습은 아직 `rejected` 쪽으로 기운 쌍을 크게 고치고, 이미 잘 가른 쌍은 거의 더 밀지 않습니다.

#### **chosen의 SFT 손실 더하기**

- **문제:** DPO 손실은 두 보상의 차이만 봅니다. 그래서 `chosen`의 확률이 내려가도 `rejected`가 더 많이 내려가면 손실이 줄어듭니다. 실무에서 흔히 보이는 모습입니다.
- **해결:** 이를 막으려고 이 교안은 `chosen`에 **SFT 단원과 같은 손실을 더합니다**(RPO(Iterative Reasoning Preference Optimization)와 비슷한 방식, [Pang et al., 2024](https://arxiv.org/abs/2404.19733)).
- **한계:** SFT 손실은 학습 쌍의 `chosen`을 붙잡을 뿐, 처음 보는 검증 쌍의 `chosen` 확률까지 지켜 주지는 않습니다(5-4).

<img src="images/09_chosen_sft_loss.png" width="1050" alt="DPO 손실만 쓰면 margin이 커지는 동안 chosen의 확률도 함께 내려갈 수 있고, chosen의 SFT 손실을 더하면 chosen의 확률을 지키며 margin을 키웁니다.">

$\displaystyle L = L_{\text{DPO}} + L_{\text{SFT}}(y_w)$

- **$L_{\text{SFT}}(y_w)$:** **SFT 단원에서 배운 손실 그대로**입니다. 정답 답변 자리에 `chosen` 답 $y_w$를 넣었을 뿐입니다.

$\displaystyle L_{\text{SFT}}(y_w) = -\frac{1}{T}\sum_{t=1}^{T} \log \pi_\theta(y_{w,t} \mid x, y_{w,<t})$

- **SFT 단원의 식:** $L = -\frac{1}{T}\sum_{t=1}^{T} \log p(y_t \mid \text{prompt}, y_{<t})$에서 $y$를 $y_w$로, $p$를 학습 중 모델 $\pi_\theta$로 바꿔 쓴 것입니다.
- **$y_{w,t}$, $y_{w,<t}$:** `chosen` 답의 $t$번째 토큰과 그 앞 토큰들입니다.
- **$T$:** 배치에 든 `chosen` 답 토큰의 수입니다. 쌍마다 따로 평균하지 않고, 배치의 `chosen` 토큰 전체를 한 번에 평균합니다.
- **`rejected`는 빼는 이유:** 더 나쁜 답을 따라 쓰게 하면 안 되므로, SFT 손실은 `chosen`에만 계산합니다.
- **학습기:** `DPOConfig`에 `loss_type=["sigmoid", "sft"]`를 넘기면 두 손실을 1 대 1로 더합니다. 그래서 학습 전 검증 손실(5-4 테이블 첫 줄의 `eval_loss`)은 0.693보다 SFT 손실만큼 큽니다.

### 2-3. beta

#### **beta**

> 2-1절 보상 식에서 **로그 확률의 차이에 곱하는 수**입니다.

- **크기의 효과:** 클수록 참조 모델과 달라지는 데 주는 **벌이 커집니다.** 이론상 학습 중 모델이 참조 모델 가까이 남도록 하는 장치이고, 실제로 얼마나 남는지는 학습률, 에폭, 더한 SFT 손실에 따라 달라집니다.
- **목적:** 선호 쪽으로 얼마나 바꿀지와 SFT 모델의 능력을 얼마나 지킬지 사이의 균형을 정합니다.

<img src="images/04_beta.png" width="1050" alt="beta가 작으면(0.1) 학습 중 모델이 참조 모델에서 멀리 움직여 선호 쪽으로 많이 바뀌고, 크면(0.5) 참조 모델 가까이 남아 덜 바뀝니다.">

| beta | 학습 중 모델 | 이 교안에서 |
|---|---|---|
| 0.1 | 벌이 작아 선호 쪽으로 더 바뀌도록 둡니다. 다른 능력이 떨어질(퇴행) 위험이 큽니다 | 권장 출발값(DPO 논문 대부분의 실험과 `DPOConfig`의 기본값) |
| 0.5 | 벌이 커서 덜 바뀌도록 막습니다. 퇴행이 덜한 대신 거절이 덜 늘 수 있습니다 | 비교 후보 |

## 3. 실습 준비와 정렬 전 답

### 3-1. 라이브러리와 경로

SFT·LoRA 단원과 같은 준비 셀입니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

### 3-2. SFT 모델 불러오기

SFT 단원에서 고른 노무 상담 SFT 모델(Tuned 3에폭)을 허깅페이스 허브에서 내려받아 SFT·LoRA 단원과 같은 float32로 불러옵니다. 허브에 그대로 올려 둔 모델이라 SFT 단원을 다시 실행하지 않아도 됩니다. 공개 저장소라 로그인이 필요 없고, 내려받을 때 나오는 `unauthenticated requests` 경고는 무시해도 됩니다.

In [ ]:
# SFT 단원에서 고른 노무 상담 SFT 모델(Tuned 3에폭)을 허깅페이스 허브에 올려 둔 저장소. 처음 실행할 때 약 3GB를 내려받습니다
# SFT 단원에서 직접 저장한 모델을 쓰려면 이 값을 그 폴더 경로(SFT 단원 폴더의 output/labor_sft_final)로 바꿉니다
SFT_MODEL_NAME = "muahseol/qwen3.5-0.8b-labor-sft"

tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_NAME)
sft_model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_NAME, dtype=torch.float32).to(device)
print("SFT 모델:", SFT_MODEL_NAME, "| 파라미터 수:", f"{sft_model.num_parameters():,}")

### 3-3. SFT 단원과 같은 함수와 데이터

시스템 프롬프트, 거절 문장, 그리디 생성 함수 `generate_replies`, `load_jsonl`은 SFT 단원 원문 그대로입니다. 학습·검증 파일도 SFT 단원과 같고, 그중 거절 답이 아닌 행(답 행)은 4-4절의 노무 질문 쌍에 씁니다.

**범위 밖 질문을 따로 모은 이유**
- SFT 데이터의 거절 질문 53개는 이미 외운 질문입니다. 고칠 대상은 처음 보는 범위 밖 질문이라, 그런 질문 50개(학습 40, 검증 10)를 `labor_out_of_scope.jsonl`에 따로 모았습니다.
- 실무에서는 서비스 로그에서 모델이 답을 지어 낸 질문을 모읍니다.

In [ ]:
# SFT 단원 데이터의 시스템 메시지와 글자까지 같은 문장. 생성할 때도 같은 문장이어야 학습한 대로 답합니다
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."

# SFT 단원의 데이터가 상담 범위 밖 질문에 단 거절 문장(DPO에서는 범위 밖 질문의 chosen 이 됩니다)
REFUSAL_ANSWER = "이 상담 챗봇이 다루는 노동법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_replies(model, questions, system_prompt, max_new_tokens=256):
    """질문을 32개씩 묶어 생성하고, 질문 순서대로 답 목록을 돌려줍니다. system_prompt=None 이면 시스템 메시지 없이 묻습니다."""
    # 여러 질문을 묶으면 짧은 질문을 패딩으로 채워 길이를 맞춥니다. 답은 오른쪽 끝에 이어 쓰므로 패딩은 왼쪽에 둡니다
    tokenizer.padding_side = "left"
    answers = []
    for start in range(0, len(questions), 32):
        # generate_reply 와 같은 chat template으로 질문을 감쌉니다
        prompts = []
        for question in questions[start:start + 32]:
            messages = []
            if system_prompt is not None:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": question})
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False))
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        # generate_reply 와 같은 그리디 생성
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                        eos_token_id=END_OF_TURN_ID, pad_token_id=tokenizer.pad_token_id)

        # 입력 길이 뒤의 새 토큰만 글자로 바꿉니다(먼저 끝난 답 뒤의 패딩은 skip_special_tokens 로 빠집니다)
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            answers.append(tokenizer.decode(new_ids, skip_special_tokens=True))

    tokenizer.padding_side = "right"  # 학습기가 쓰는 오른쪽 패딩으로 되돌립니다
    return answers


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(SFT 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
# SFT 단원에서 원본 단위로 나눠 저장한 노무 상담 학습, 검증 파일
train_rows = load_jsonl(data_dir / "본실습" / "labor_train.jsonl")
val_rows = load_jsonl(data_dir / "본실습" / "labor_val.jsonl")

# 거절 키워드로 답 행(노무 질문)만 남깁니다(키워드는 고정 거절 문장에만 씁니다)
answer_rows = []
for row in train_rows:
    if "답할 수 없" not in row["answer"]:
        answer_rows.append(row)
val_answer_rows = []
for row in val_rows:
    if "답할 수 없" not in row["answer"]:
        val_answer_rows.append(row)

# DPO용으로 따로 모은 범위 밖 질문을 split 열로 학습과 검증으로 나눕니다
labor_out_of_scope_rows = load_jsonl(data_dir / "본실습" / "labor_out_of_scope.jsonl")
train_refusal_questions = []
val_refusal_questions = []
for row in labor_out_of_scope_rows:
    if row["split"] == "train":
        train_refusal_questions.append(row["question"])
    else:
        val_refusal_questions.append(row["question"])

print("답 행(학습, 검증):", len(answer_rows), len(val_answer_rows),
      "| 범위 밖 질문(학습, 검증):", len(train_refusal_questions), len(val_refusal_questions))

### 3-4. 정렬 전 답

정렬 전후에 똑같이 물을 질문 다섯 개로 SFT 모델의 답을 먼저 봅니다. 정렬 모델의 답은 5-5절에서 같은 테이블에 열로 더합니다.

| 종류 | 개수 | 보는 것 |
|---|---|---|
| 범위 밖 질문(검증) | 2 | DPO 학습에 쓰지 않은 범위 밖 질문을 거절하는지 |
| 노무 질문(검증) | 1 | 학습에 쓰지 않은 노무 질문에 거절하지 않고 답하는지(과잉 거절) |
| 노무 질문(학습) | 1 | SFT로 배운 답을 지키는지 |
| 일반 질문 | 1 | 시스템 프롬프트 없이 상식 답을 지키는지 |

In [ ]:
# 노무와 상관없는 상식 질문(정답 답변은 reference_answer)
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")

# 정렬 전후에 똑같이 물을 질문 다섯 개와 그 종류. 범위 밖 질문은 검증 질문 앞 두 개, 일반 질문은 SFT 단원에서 만든 SFT 모델이 노무 답을 섞었던 명절 질문입니다
check_questions = [val_refusal_questions[0], val_refusal_questions[1], val_rows[39]["question"],
                   train_rows[300]["question"], general_rows[15]["question"]]
check_kinds = ["범위 밖 질문(검증)", "범위 밖 질문(검증)", "노무 질문(검증)", "노무 질문(학습)", "일반 질문"]

# 질문마다 정답 답변: 범위 밖 질문은 거절 문장, 노무 질문은 answer, 일반 질문은 reference_answer
reference_answers = [REFUSAL_ANSWER, REFUSAL_ANSWER, val_rows[39]["answer"],
                     train_rows[300]["answer"], general_rows[15]["reference_answer"]]


def answer_check_questions(model):
    """질문 다섯 개에 model이 그리디 생성으로 한 답을 같은 순서의 리스트로 돌려줍니다."""
    # 앞 네 질문은 서비스와 같은 노무 상담 시스템 프롬프트로 묻습니다
    answers = generate_replies(model, check_questions[:4], SYSTEM_PROMPT)

    # 일반 질문은 시스템 프롬프트 없이 묻습니다(노무 프롬프트로 물으면 거절이 상식을 잃은 것인지 가를 수 없습니다)
    answers += generate_replies(model, check_questions[4:], None)
    return answers

In [ ]:
# 종류, 질문, 정답 답변, SFT 모델 답변을 한 테이블에 둡니다. 정렬 모델의 답변은 뒤에서 열로 더합니다
check_table = pd.DataFrame({
    "종류": check_kinds,
    "질문": check_questions,
    "정답 답변": reference_answers,
    "SFT 모델 답변": answer_check_questions(sft_model),
})
display(check_table)

**결과 읽기**
- **범위 밖 질문:** 두 질문 모두 거절하지 않고 답을 지어 냅니다. 채용서류 질문에는 '근로기준법 제43조의8'처럼 질문과 맞지 않는 조문을 끌어와 답하고, 발표 질문에는 노무와 상관없는 조언을 어색한 문장으로 답합니다.
- **노무 질문과 일반 질문:** 휴게시간은 정답 답변처럼 '30분 이상'으로 맞히지만, 배우자 유산 휴가는 '20일'로 틀립니다(정답은 5일, 최초 3일 유급). 명절 질문에는 추석이라고 답하지 못합니다.

## 4. 선호 데이터 만들기

### 4-1. 범위 밖 쌍과 노무 질문 쌍

선호 쌍을 두 종류로 만들어 함께 학습합니다. 범위 밖 쌍은 거절을 가르치고, 노무 질문 쌍은 노무 질문까지 거절하지 않게 막습니다.

<img src="images/02_preference_pair.png" width="1050" alt="범위 밖 쌍은 거절 문장을 chosen, SFT 모델이 지어 낸 답을 rejected로 두고, 노무 질문 쌍은 학습 데이터의 답을 chosen, 거절 문장을 rejected로 두어 과잉 거절을 막습니다.">

어느 답이 나은지 정하는 방법은 세 가지입니다. 이 교안은 범위 밖 질문의 정답이 고정 거절 문장이라 규칙으로 정합니다.

| 방법 | 누가 정하나 | 쓰는 때 |
|---|---|---|
| 사람 라벨 | 사람이 두 답을 비교해 고릅니다 | 말투, 친절함처럼 기준이 사람 판단에 달린 답. 비용이 큽니다 |
| 판사 모델(AI 피드백) | LLM 판사가 두 답을 비교해 고릅니다 | 쌍을 많이 모아야 할 때. 판사의 판정을 표본으로 사람이 확인합니다 |
| 규칙 | 정해진 문장이나 검사로 정합니다 | 답이 고정 문장이거나 맞는지 검사할 수 있을 때(이 교안의 거절 문장) |

- **on-policy(학습을 시작하는 모델이 직접 낸 답으로 만든 데이터):** 여기서는 범위 밖 쌍의 `rejected`만 SFT 모델이 낸 답이고, 학습 전에 한 번 뽑아 둡니다. 모델이 내지도 않는 답은 확률이 이미 낮아서, 더 낮춰도 실제 답이 거의 바뀌지 않습니다.

### 4-2. SFT 모델의 답 여러 개 뽑기

그리디 생성은 확률이 가장 큰 토큰만 골라 같은 질문에 늘 같은 답을 냅니다. 샘플링(sampling)은 확률에 따라 토큰을 뽑아 다른 답을 낼 수 있으므로, 질문마다 4개를 뽑으면 그리디 답 하나보다 지어 낸 답을 더 많이 찾습니다.

<img src="images/07_sampling_pairs.png" width="1050" alt="범위 밖 질문마다 SFT 모델의 답을 4개 샘플링해 거절 문장이 없는 첫 답을 rejected, 고정 거절 문장을 chosen으로 두고, 네 답이 모두 거절이면 쌍을 만들지 않습니다.">

In [ ]:
def sample_replies(model, questions, system_prompt, num_samples=4, max_new_tokens=256):
    """질문마다 샘플링으로 답을 num_samples 개 만들어, 질문 순서대로 답 리스트를 담은 리스트를 돌려줍니다."""
    all_samples = []
    for question in questions:
        # generate_replies 와 같은 chat template으로 질문 하나를 감쌉니다
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # 확률에 따라 토큰을 뽑아(샘플링) 같은 질문에서 서로 다른 답을 num_samples 개 만듭니다
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,                        # 그리디 대신 확률대로 뽑습니다
                temperature=1.0,                       # 확률을 키우거나 줄이지 않습니다(작을수록 그리디에 가까움)
                top_p=0.95,                            # 확률 합 95% 안의 토큰에서만 뽑아 아주 드문 토큰을 뺍니다
                num_return_sequences=num_samples,      # 한 질문에서 답을 이 개수만큼 만듭니다
                eos_token_id=END_OF_TURN_ID,
                pad_token_id=tokenizer.pad_token_id,
            )

        # 답마다 입력 길이 뒤의 새 토큰만 글자로 바꿉니다
        samples = []
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            samples.append(tokenizer.decode(new_ids, skip_special_tokens=True))
        all_samples.append(samples)
    return all_samples

In [ ]:
# 시드를 고정해 다시 실행해도 같은 답을 뽑습니다. 질문마다 답 4개의 리스트가 돌아옵니다
torch.manual_seed(0)
train_samples = sample_replies(sft_model, train_refusal_questions, SYSTEM_PROMPT)
val_samples = sample_replies(sft_model, val_refusal_questions, SYSTEM_PROMPT)

### 4-3. 범위 밖 쌍 만들기

질문마다 샘플링 답 4개 가운데 거절 키워드가 없는 첫 답을 `rejected`, 거절 문장을 `chosen`으로 둡니다. 4개가 모두 거절이면 이미 거절하는 질문이라 고칠 것이 없으므로 쌍을 만들지 않습니다.

In [ ]:
# 학습과 검증에 같은 규칙을 적용합니다
out_of_scope_pairs = {"train": [], "val": []}
skipped_counts = {"train": 0, "val": 0}
for split, questions, samples in [("train", train_refusal_questions, train_samples), ("val", val_refusal_questions, val_samples)]:
    for question, answers in zip(questions, samples):
        # 거절 키워드가 없고 비어 있지 않은 첫 답이 rejected(SFT 모델이 지어 낸 답)
        rejected = None
        for answer in answers:
            if "답할 수 없" not in answer and answer.strip() != "":
                rejected = answer
                break

        # 네 답이 모두 거절이면 쌍을 만들지 않고 수만 셉니다
        if rejected is None:
            skipped_counts[split] += 1
        else:
            out_of_scope_pairs[split].append({"question": question, "chosen": REFUSAL_ANSWER, "rejected": rejected})

print("범위 밖 쌍 | 학습:", len(out_of_scope_pairs["train"]), "| 검증:", len(out_of_scope_pairs["val"]),
      "| 네 답이 모두 거절이라 건너뛴 질문(학습, 검증):", skipped_counts["train"], skipped_counts["val"])

In [ ]:
# 범위 밖 학습 쌍의 앞 다섯 개(답이 길어 전체 대신 앞부분만 봅니다)
out_of_scope_table = pd.DataFrame(out_of_scope_pairs["train"])
display(out_of_scope_table.head())

**결과 읽기**
- **쌍 수:** 범위 밖 학습 쌍 40개, 검증 쌍 9개이고, 검증 질문 1개는 네 답이 모두 거절이라 건너뛰었습니다. 건너뛴 질문이 많을수록 SFT 모델이 처음 보는 범위 밖 질문도 이미 거절한다는 뜻입니다.
- **`rejected`:** 노동조합 설립 질문에 '노동위원회에 신고해야 합니다'처럼 노동법을 끌어온 그럴듯한 답도 있고, 빨래 질문처럼 문장이 깨진 답도 있습니다. 어느 쪽이든 SFT 모델이 실제로 낸 답입니다.

### 4-4. 노무 질문 쌍 만들기

범위 밖 쌍만 학습하면 모델이 거절 문장 자체를 자주 내게 되어 노무 질문까지 거절할 수 있습니다(과잉 거절). 그래서 노무 질문에서는 학습 데이터의 답을 `chosen`, 거절 문장을 `rejected`로 둔 쌍을 함께 넣습니다.

- **범위 밖 쌍의 절반만:** 같은 거절 문장이 범위 밖 쌍에서는 확률을 올리는 쪽(`chosen`), 노무 질문 쌍에서는 내리는 쪽(`rejected`)입니다. 범위 밖 질문에서 올리는 변화와 노무 질문에서 내리는 변화가 같은 파라미터를 반대로 당기므로, 두 쌍이 같은 수면 범위 밖 질문에서도 거절 문장이 잘 오르지 않을 수 있습니다. 그래서 노무 질문 쌍은 절반만 넣습니다.
- **on-policy가 아닌 쌍:** 이 쌍의 거절 문장은 SFT 모델이 지금 노무 질문에 내는 답이 아닙니다. 거절이 노무 질문으로 번지는 것을 막는 용도입니다.
- **길이 편향(length bias):** `chosen`이 대체로 `rejected`보다 길면 DPO가 품질 대신 '길게 답하기'를 배울 수 있습니다. 범위 밖 쌍은 `rejected` 길이가 제각각이라 `chosen`이 짧을 때도 길 때도 있고, 노무 질문 쌍은 `chosen`이 대체로 깁니다.
- **고르는 행:** 교안 02 평가에 쓰는 행과 겹치지 않는 행입니다.

<img src="images/16_refusal_cancel.png" width="1050" alt="같은 거절 문장이 범위 밖 쌍에서는 chosen, 노무 질문 쌍에서는 rejected라 두 쌍이 같은 수면 올리고 내리는 변화가 상쇄되고, 2 : 1로 넣으면 거절 문장의 확률이 오릅니다.">

In [ ]:
# 학습: 답 행의 앞 13행을 건너뛰고 54행 간격으로 20행. chosen 은 학습 데이터의 답, rejected 는 거절 문장
labor_pairs = {"train": [], "val": []}
for row in answer_rows[13::54][:20]:
    labor_pairs["train"].append({"question": row["question"], "chosen": row["answer"], "rejected": REFUSAL_ANSWER})

# 검증: 검증 답 행에서 29행 간격으로 5행(범위 밖 검증 질문 수의 절반)
for row in val_answer_rows[::29][:5]:
    labor_pairs["val"].append({"question": row["question"], "chosen": row["answer"], "rejected": REFUSAL_ANSWER})

print("노무 질문 쌍 | 학습:", len(labor_pairs["train"]), "| 검증:", len(labor_pairs["val"]))

### 4-5. 대화형 Dataset으로 바꾸기

`DPOTrainer`는 `prompt`, `chosen`, `rejected`를 메시지 리스트(대화형 행)로 받으면, prompt에 chat template을 씌우고 뒤에 두 답을 각각 이어 붙입니다. 생성할 때와 같은 설정은 `chat_template_kwargs`로 넘깁니다.

In [ ]:
def to_preference(question, chosen, rejected, system_prompt):
    """질문 하나와 두 답을 DPOTrainer 가 받는 대화형 선호 쌍 한 행으로 바꿉니다."""
    return {
        "prompt": [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
        "chosen": [{"role": "assistant", "content": chosen}],      # 더 나은 답
        "rejected": [{"role": "assistant", "content": rejected}],  # 더 나쁜 답
        "chat_template_kwargs": {"enable_thinking": False},         # 생성할 때와 같은 빈 생각 블록을 붙입니다
    }

In [ ]:
from datasets import Dataset

# 학습 쌍(범위 밖 + 노무 질문)과 검증 쌍을 대화형 행으로 바꿉니다. 섞기는 학습기가 에폭마다 합니다
train_pairs = Dataset.from_list(
    [to_preference(pair["question"], pair["chosen"], pair["rejected"], SYSTEM_PROMPT) for pair in out_of_scope_pairs["train"] + labor_pairs["train"]]
)
val_pairs = Dataset.from_list(
    [to_preference(pair["question"], pair["chosen"], pair["rejected"], SYSTEM_PROMPT) for pair in out_of_scope_pairs["val"] + labor_pairs["val"]]
)
print("학습 쌍:", len(train_pairs), "| 검증 쌍:", len(val_pairs))

## 5. DPO 학습(beta 0.1)

### 5-1. LoRA 설정과 참조 모델

어댑터는 LoRA 단원과 같은 설정으로 SFT 모델에 붙입니다. 0.8B 모델이라 QLoRA 단원과 달리 양자화하지 않고 float32 그대로 두며, 학습 계산은 SFT·LoRA 단원처럼 `DPOConfig` 기본값(bf16 자동 혼합 정밀도)으로 합니다.

- **참조 모델 = 어댑터를 끈 SFT 모델:** `DPOTrainer`에 `peft_config`를 넘기면 학습기가 SFT 모델에 새 어댑터를 붙이고, 참조 모델의 확률은 같은 모델에서 어댑터만 끄고 계산합니다. 참조 모델을 따로 불러오지 않아 GPU 메모리에 모델이 하나만 올라갑니다.

<img src="images/05_lora_reference.png" width="1050" alt="LoRA로 DPO를 하면 SFT 모델에 붙인 어댑터를 켠 모델이 학습 중 모델이고, 어댑터를 끈 같은 모델이 참조 모델이라 모델을 하나만 올립니다.">

**DPO와 LoRA, 풀 파인튜닝**

DPO는 풀 파인튜닝으로도, LoRA로도 학습합니다. 주로 쓸 수 있는 GPU 메모리에 따라 고릅니다.

| 방식 | 쓰인 예 | 참조 모델의 메모리 |
|---|---|---|
| 풀 파인튜닝 | DPO 논문의 실험, [Zephyr-7B-beta](https://arxiv.org/abs/2310.16944), [Tulu 3](https://arxiv.org/abs/2411.15124)이 풀 파인튜닝으로 DPO를 학습했습니다 | 같은 크기의 모델을 한 벌 더 올리거나, 참조 모델의 로그 확률을 학습 전에 미리 계산해 둡니다 |
| LoRA, QLoRA | TRL이 공식으로 지원하는 메모리를 아끼는 방법 | 따로 올리지 않습니다(위 그림) |

- Hugging Face alignment-handbook의 Zephyr 레시피는 풀 파인튜닝과 QLoRA 설정을 둘 다 제공합니다.
- 0.8B 모델이라 풀 파인튜닝 DPO도 GPU 한 장으로 할 수 있습니다. 이 교안이 LoRA를 쓰는 이유는 참조 모델을 따로 올리지 않아도 되고, 에폭마다 작은 어댑터만 저장하면 되기 때문입니다. 두 방식의 학습률 차이는 5-2절에서 봅니다.

In [ ]:
from peft import LoraConfig

# LoRA 단원과 같은 설정. 어댑터를 붙이는 일은 DPOTrainer 가 합니다
lora_config = LoraConfig(
    r=8,                          # B, A의 가운데 차원(BA 의 rank 상한)
    lora_alpha=16,                # 배율 = lora_alpha / r = 2.0
    target_modules="all-linear",  # lm_head를 뺀 모든 선형 층
    lora_dropout=0.05,            # LoRA 단원과 같은 값. 학습기가 드롭아웃을 끄므로(DPOConfig disable_dropout 기본값 True) 학습 중에는 적용되지 않습니다
    task_type="CAUSAL_LM",        # 다음 토큰을 예측하는 생성 모델
)

### 5-2. 학습 설정

`DPOConfig`는 SFT·LoRA 단원의 `SFTConfig`와 같은 학습 인자에 DPO 인자(`beta`, `loss_type` 등)를 더한 설정입니다. 인자마다 뜻과 이 값을 쓰는 이유를 코드 주석에 적었습니다. 학습 기록은 MLflow 실험 `labor-dpo`에 남깁니다.

**학습률**
- **기준값:** TRL의 DPO 예제와 [TRL PEFT 가이드](https://huggingface.co/docs/trl/main/en/peft_integration)는 풀 파인튜닝에 5e-7, LoRA에 그 10배인 5e-6을 씁니다. LoRA가 약 10배 큰 학습률을 쓰는 것은 LoRA 단원과 같은 기준입니다.
- **SFT보다 작게:** 풀 파인튜닝끼리 비교하면 DPO 예제의 5e-7은 SFT 단원의 학습률(5e-6)보다 작습니다. 이미 배운 답의 확률을 조금 옮기는 일이라, 크게 움직이면 퇴행이 커지기 때문입니다.
- **이 교안의 값:** 갱신이 모두 45번 이하(학습 쌍 최대 60개)로 예제보다 훨씬 적어, 예제 LoRA 값의 10배인 5e-5를 씁니다(LoRA 단원의 2e-4보다는 작습니다).

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("labor-dpo")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

In [ ]:
from trl import DPOConfig, DPOTrainer

# SFT·LoRA 단원의 SFTConfig 와 같은 모양의 학습 인자 + DPO 인자(beta, loss_type)
dpo_config = DPOConfig(
    output_dir=str(output_dir / "labor_dpo_beta0.1"),  # 에폭마다 checkpoint-<갱신 횟수> 폴더가 생깁니다(교안 02가 불러옵니다)
    beta=0.1,                                          # 보상 = beta * 로그 확률의 차이. DPO 논문과 DPOConfig 기본값
    loss_type=["sigmoid", "sft"],                      # DPO 손실 + chosen 의 SFT 손실. 기본값 ["sigmoid"]는 DPO 손실만 씁니다
    learning_rate=5e-5,                                # 위 '학습률'의 값. DPOConfig 기본값은 1e-6
    num_train_epochs=3,                                # 에폭마다 어댑터를 저장해 교안 02에서 채점해 고릅니다(기본값과 같음)
    per_device_train_batch_size=4,                     # 한 번 갱신에 쓰는 쌍 수. SFT·LoRA 단원과 같음(기본값 8)
    per_device_eval_batch_size=4,                      # 검증 쌍도 같은 배치로 계산합니다
    seed=0,                                            # 쌍을 섞는 순서를 고정해 다시 실행해도 같은 순서로 학습합니다
    gradient_checkpointing=False,                      # DPOConfig 기본은 켬. 0.8B 모델은 메모리가 충분해 끄고 빠르게 계산합니다(SFT·LoRA 단원과 같음)
    max_length=1024,                                   # prompt + 답의 토큰 상한(SFT·LoRA 단원과 같음)
    eval_strategy="epoch",                             # 에폭이 끝날 때마다 검증 쌍의 손실과 보상을 기록합니다
    save_strategy="epoch",                             # 에폭이 끝날 때마다 어댑터를 저장합니다
    save_only_model=True,                              # 옵티마이저 상태 없이 어댑터만 저장합니다
    logging_steps=5,                                   # 갱신이 모두 45번 이하라 5번 갱신마다 학습 기록을 남깁니다
    eval_on_start=True,                                # 학습 전 검증 기록(보상 0, margin 0)을 기준선으로 남깁니다
    report_to=["mlflow"],                              # 손실과 보상 지표를 MLflow run 에 남깁니다
    run_name="dpo_beta_0.1",                           # MLflow 에 보일 run 이름(beta 가 드러나게)
)

# peft_config 를 넘기면 학습기가 sft_model 에 어댑터를 붙이고, 어댑터를 끈 같은 모델로 참조 모델의 로그 확률을 계산합니다
trainer = DPOTrainer(
    model=sft_model,
    args=dpo_config,
    train_dataset=train_pairs,
    eval_dataset=val_pairs,
    processing_class=tokenizer,
    peft_config=lora_config,
)

### 5-3. 학습

In [ ]:
# 쌍 수, SFT 모델 폴더, LoRA 설정을 run에 남기고 학습합니다. 갱신 횟수 = 올림(학습 쌍 수 / 4) * 3
with mlflow.start_run(run_name=dpo_config.run_name):
    mlflow.log_params({"sft_model": SFT_MODEL_NAME, "out_of_scope_pairs": len(out_of_scope_pairs["train"]),
                       "labor_pairs": len(labor_pairs["train"]), "val_pairs": len(val_pairs),
                       "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha})
    trainer.train()

### 5-4. 학습 기록 읽기

`DPOTrainer`는 손실과 함께 보상 지표를 기록합니다. 검증 기록은 같은 이름 앞에 `eval_`이 붙습니다.

<img src="images/06_reward_metrics.png" width="1050" alt="DPO 학습 기록의 네 값: rewards/chosen, rewards/rejected, rewards/margins(두 보상의 차이), rewards/accuracies(chosen 보상이 더 큰 쌍의 비율).">

In [ ]:
# 학습기 기록에서 학습 줄(loss 가 있는 줄)과 검증 줄(eval_loss 가 있는 줄)을 나눕니다
log_table = pd.DataFrame(trainer.state.log_history)
train_log = log_table.dropna(subset=["loss"])
eval_log = log_table.dropna(subset=["eval_loss"])

# 에폭별 검증 손실과 보상 지표(맨 위는 학습 전)
reward_columns = ["eval_rewards/chosen", "eval_rewards/rejected", "eval_rewards/margins", "eval_rewards/accuracies"]
display(eval_log[["step", "epoch", "eval_loss"] + reward_columns].round(4))

In [ ]:
# 왼쪽은 margins, 오른쪽은 accuracies. 학습 기록(파랑)은 5번 갱신마다, 검증 기록(빨강)은 에폭마다 있습니다
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_log["epoch"], train_log["rewards/margins"], marker="o", color="#2563eb", label="학습 (rewards/margins)")
axes[0].plot(eval_log["epoch"], eval_log["eval_rewards/margins"], marker="o", color="#dc2626", label="검증 (eval_rewards/margins)")
axes[0].set_title("margin = chosen 보상 - rejected 보상")
axes[0].set_xlabel("에폭 (epoch)")
axes[0].set_ylabel("margin")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(train_log["epoch"], train_log["rewards/accuracies"], marker="o", color="#2563eb", label="학습 (rewards/accuracies)")
axes[1].plot(eval_log["epoch"], eval_log["eval_rewards/accuracies"], marker="o", color="#dc2626", label="검증 (eval_rewards/accuracies)")
axes[1].set_title("chosen 보상이 더 큰 쌍의 비율")
axes[1].set_xlabel("에폭 (epoch)")
axes[1].set_ylabel("accuracies")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.suptitle("DPO 학습 기록 (beta 0.1)")
plt.tight_layout()
plt.show()

**테이블과 그래프 읽기**
- **학습 전(0에폭):** 두 모델이 같아 보상과 margin이 모두 0이고, accuracies도 0입니다(두 보상이 같으면 '더 큰 쌍'에 들지 않습니다). 손실은 0.693에 `chosen`의 SFT 손실을 더한 값입니다.
- **학습 쌍(파랑):** 학습 margins는 3에폭에 약 2.7까지 커지고, 학습 accuracies는 1.0에 이릅니다. 학습 쌍의 두 답은 확실히 가르게 됐습니다.
- **검증 쌍(빨강):** 검증 margins는 0.016, 0.014, -0.009로 0 근처에 머물고, 검증 accuracies도 0.56 그대로입니다. 3에폭에는 eval_rewards/chosen(-0.087)이 rejected(-0.078)보다 더 내려가 margin이 음수가 됐습니다. `chosen`의 SFT 손실을 더했어도 처음 보는 검증 쌍의 `chosen`은 내려갈 수 있습니다(2-2의 '한계').
- **두 기록이 갈리는 이유:** 학습 쌍 60개를 3에폭 반복하며 그 쌍의 답을 외운 모습입니다. 검증 쌍에서는 `chosen`과 `rejected`가 함께 내려가(보상 -0.087, -0.078) margin이 0 근처에 머물렀습니다. 보상은 beta 0.1을 곱한 값이라, 로그 확률로는 각각 약 0.87, 0.78 내려간 것입니다.
- **실행 환경에 따른 차이:** 이 값은 강사 실행 환경(맥 MPS, trl 1.16, transformers 5.17)의 결과입니다. 검증 쌍이 14개뿐이고 margin이 0 근처라, 환경이 바뀌면 검증 margin의 부호까지 달라질 수 있습니다.
- **손실, accuracies와 실제 답:** 손실과 accuracies는 정해 둔 두 답의 차이와 순서만 봅니다. 검증 accuracies가 0.56에 머물러도 생성한 답이 거절로 바뀔 수 있고(5-5), 학습 accuracies가 1.0이어도 새 질문의 답은 바뀌지 않을 수 있습니다. 그래서 모델이 새 질문에 거절하는지는 생성한 답으로 확인합니다.

### 5-5. 정렬 전후 답

SFT 모델에 물은 질문 다섯 개를 beta 0.1 정렬 모델(3에폭 끝)에 같은 그리디 생성으로 묻고, 같은 테이블에 열로 더합니다. 생성 전에 `eval()`로 평가 모드로 바꿉니다.

In [ ]:
# 평가 모드로 바꾸고, 같은 다섯 질문을 beta 0.1 정렬 모델에 물어 열로 더합니다
trainer.model.eval()
check_table["정렬 모델 답변"] = answer_check_questions(trainer.model)
display(check_table)

**결과 읽기**
- **범위 밖 질문:** 발표 질문에는 거절 문장 그대로 답합니다. 검증 margin은 거의 움직이지 않았어도 이 질문의 답은 거절로 바뀌었습니다. 채용서류 질문에는 여전히 '고용노동부장관은 …'으로 답을 지어 냅니다.
- **노무 질문과 일반 질문:** 휴게시간 질문에는 거절하지 않고 같은 내용(30분 이상)으로 답합니다. 배우자 유산 휴가는 '20일'에서 '10일, 최초 5일 유급'으로 바뀌었지만 여전히 틀립니다(정답은 5일, 최초 3일 유급). 이 교안의 선호 쌍은 거절할지만 가르치므로, 틀린 휴가 일수를 바로잡는 신호가 없습니다(20일이 10일로 바뀐 것은 부수 변화입니다). 명절 질문의 답도 문장만 바뀌고 여전히 추석이라고 답하지 못합니다.
- 질문 다섯 개라 경향만 보입니다. 거절률, 과잉 거절률, 일반 능력은 교안 02에서 평가셋으로 계산합니다.

## 6. beta 비교(beta 0.5)

### 6-1. 새 SFT 모델에 새 어댑터

앞의 `sft_model`에는 beta 0.1 어댑터가 붙어 있으므로, SFT 모델을 새로 불러와 beta만 바꿔 학습합니다. 그 전에 LoRA 단원처럼 beta 0.1 학습기와 모델을 지워 GPU 메모리를 비웁니다(답은 테이블에, 어댑터는 에폭마다 저장해 두었습니다).

In [ ]:
import gc

# 앞 학습의 학습기와 모델을 지웁니다(한 번만 실행)
del trainer, sft_model
gc.collect()              # 더 이상 쓰지 않는 객체를 메모리에서 정리합니다
torch.cuda.empty_cache()  # PyTorch가 잡아 두었던 빈 GPU 메모리를 돌려줍니다

In [ ]:
# 같은 SFT 모델을 새로 불러옵니다(학습기가 여기에 새 어댑터를 붙입니다)
beta_high_model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_NAME, dtype=torch.float32).to(device)

# dpo_config 와 같은 값입니다. 다른 줄에만 '다름'을 적었습니다
beta_high_config = DPOConfig(
    output_dir=str(output_dir / "labor_dpo_beta0.5"),  # 다름
    beta=0.5,                                          # 다름: 참조 모델과 달라지는 데 주는 벌을 키웁니다
    loss_type=["sigmoid", "sft"],
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    seed=0,
    gradient_checkpointing=False,
    max_length=1024,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,
    logging_steps=5,
    eval_on_start=True,
    report_to=["mlflow"],
    run_name="dpo_beta_0.5",                           # 다름
)
beta_high_trainer = DPOTrainer(
    model=beta_high_model,
    args=beta_high_config,
    train_dataset=train_pairs,
    eval_dataset=val_pairs,
    processing_class=tokenizer,
    peft_config=lora_config,
)

In [ ]:
# 같은 쌍, 같은 갱신 횟수로 학습합니다. run 에 남기는 값도 같습니다
with mlflow.start_run(run_name=beta_high_config.run_name):
    mlflow.log_params({"sft_model": SFT_MODEL_NAME, "out_of_scope_pairs": len(out_of_scope_pairs["train"]),
                       "labor_pairs": len(labor_pairs["train"]), "val_pairs": len(val_pairs),
                       "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha})
    beta_high_trainer.train()

### 6-2. 두 beta의 학습 기록

보상에는 beta가 곱해져 있어, 로그 확률이 똑같이 움직여도 beta 0.5의 margin은 beta 0.1의 5배로 나옵니다. 그래서 margin을 beta로 나눈 값으로 비교합니다. 이 값은 `chosen`의 로그 확률 변화에서 `rejected`의 로그 확률 변화를 뺀 값으로, beta와 상관없이 비교할 수 있습니다. 검증 accuracies도 에폭별로 나란히 놓습니다.

In [ ]:
# beta 0.5 학습의 검증 줄
beta_high_log_table = pd.DataFrame(beta_high_trainer.state.log_history)
beta_high_eval_log = beta_high_log_table.dropna(subset=["eval_loss"])

# 에폭별 검증 accuracies 와 margin / beta (0에폭은 학습 전)
beta_compare_table = pd.DataFrame({
    "에폭": eval_log["epoch"].tolist(),
    "beta 0.1 accuracies": eval_log["eval_rewards/accuracies"].tolist(),
    "beta 0.5 accuracies": beta_high_eval_log["eval_rewards/accuracies"].tolist(),
    "beta 0.1 margin / beta": (eval_log["eval_rewards/margins"] / 0.1).tolist(),
    "beta 0.5 margin / beta": (beta_high_eval_log["eval_rewards/margins"] / 0.5).tolist(),
})
display(beta_compare_table.round(4))

**테이블 읽기**
- 검증 margin / beta는 에폭마다 beta 0.1이 더 큽니다(1에폭 0.16 대 -0.15, 3에폭 -0.09 대 -0.38). beta 0.5는 1에폭부터 음수라, 검증 쌍에서는 `rejected` 쪽으로 기울었습니다.
- accuracies는 1에폭에 beta 0.5가 낮고(0.44 대 0.56) 2, 3에폭에는 둘 다 0.56입니다. accuracies는 순서만 세므로, 얼마나 움직였는지는 margin / beta로 봅니다.
- 두 beta 모두 검증 값이 0 근처라, 이 학습 기록만으로는 어느 쪽이 나은지 말할 수 없습니다. 어느 beta, 어느 에폭의 어댑터를 쓸지는 교안 02에서 미리 정한 규칙과 평가셋 지표로 정합니다.
- **MLflow 화면:** SFT 단원과 같은 명령을 단원 폴더에서 실행하면(`uv run mlflow server --backend-store-uri sqlite:///output/mlflow.db --port 5050`) 실험 `labor-dpo`의 두 run에서 `eval_rewards/margins`, `eval_rewards/accuracies`를 겹쳐 볼 수 있습니다.

두 학습 모두 에폭마다 `checkpoint-<갱신 횟수>` 폴더에 어댑터가 저장되었습니다. 본실습이 끝났으므로 beta 0.5 학습기와 모델을 지워 GPU 메모리를 비웁니다.

In [ ]:
# beta 0.5 학습기와 모델을 지웁니다(한 번만 실행)
del beta_high_trainer, beta_high_model
gc.collect()
torch.cuda.empty_cache()

## 7. 🖐️ 함께 따라하기: 전월세 상담 모델 정렬하기

본실습의 선호 쌍 만들기와 DPO 학습을 SFT 단원의 함께 따라하기에서 만든 전월세 상담 SFT 모델에 적용해, 범위 밖 질문을 거절하도록 정렬하고 어댑터를 저장합니다. 교안 02 함께 따라하기가 이 어댑터를 평가합니다.

**본실습과 다른 점**
- **SFT 모델:** SFT 단원의 교안 01 함께 따라하기에서 만든 전월세 SFT 모델(`lease_lr5e-6`)을 허브에 올린 `muahseol/qwen3.5-0.8b-lease-sft`입니다. 시스템 프롬프트는 `SYSTEM_PROMPT_LEASE`입니다.
- **범위 밖 질문:** 전월세 학습 데이터에는 거절 답 행이 없어 SFT 모델이 거절 문장을 배운 적이 없습니다. 그래서 본실습보다 많은 범위 밖 질문 80개를 `data/따라하기/lease_out_of_scope.jsonl`에 따로 준비했습니다(`split` 열: 학습 64, 검증 16).
- **거절 문장:** `LEASE_REFUSAL_ANSWER`. 노무 거절 문장과 같은 모양이고 거절 키워드 `답할 수 없`도 같습니다.
- **전월세 질문 쌍:** 학습 쌍은 본실습처럼 범위 밖 쌍의 절반입니다. `lease_train` 4행 간격 32행(학습), `lease_val` 8행 간격 4행(검증)이고, `chosen`은 학습 데이터의 답, `rejected`는 거절 문장입니다.
- **학습률:** 1e-4로 본실습의 2배입니다. 거절 문장을 처음 배우는 모델이라 본실습 학습률로는 3에폭이 지나도 처음 보는 범위 밖 질문을 거절하지 않았습니다.
- **학습:** beta 0.1 하나로 3에폭, 중간 체크포인트는 저장하지 않고(`save_strategy="no"`) 끝난 어댑터를 `output/lease_dpo_adapter`에 저장합니다.
- **나머지 설정:** `loss_type`, 배치, `max_length`, 검증 기록은 본실습 `dpo_config`와 같고, 다른 줄에만 코드 주석으로 '다름'을 적었습니다. run 이름은 `lease_dpo_beta_0.1`이고, 본실습 run과 섞이지 않게 따로 만든 실험 `lease-dpo`에 남깁니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 빈칸은 `rejected`를 고르는 조건, `chosen`과 `rejected`, `beta`, `peft_config`에만 있습니다. 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.
- 이 절은 앞 셀이 만든 함수와 설정을 씁니다. 3절부터 5-2절까지 차례로 실행한 뒤(5-3 학습 셀만 건너뜀) 이 절을 실행합니다.

### 7-1. 데이터와 SFT 모델 읽기

In [ ]:
# 전월세 데이터의 시스템 메시지와 글자까지 같은 문장, 전월세 상담의 거절 문장
SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."
LEASE_REFUSAL_ANSWER = "이 상담 챗봇이 다루는 주택임대차보호법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# SFT 단원의 따라하기와 같은 전월세 학습, 검증 파일과 따로 준비한 범위 밖 질문
lease_train_rows = load_jsonl(data_dir / "따라하기" / "lease_train.jsonl")
lease_val_rows = load_jsonl(data_dir / "따라하기" / "lease_val.jsonl")
lease_out_of_scope_rows = load_jsonl(data_dir / "따라하기" / "lease_out_of_scope.jsonl")

# 범위 밖 질문을 split 열로 학습과 검증으로 나눕니다
lease_train_refusal_questions = []
lease_val_refusal_questions = []
for row in lease_out_of_scope_rows:
    if row["split"] == "train":
        lease_train_refusal_questions.append(row["question"])
    else:
        lease_val_refusal_questions.append(row["question"])
print("전월세 학습:", len(lease_train_rows), "| 검증:", len(lease_val_rows),
      "| 범위 밖 질문(학습, 검증):", len(lease_train_refusal_questions), len(lease_val_refusal_questions))

# SFT 단원의 함께 따라하기에서 만든 전월세 SFT 모델을 허브에서 내려받습니다(토크나이저는 같은 Qwen 토크나이저라 그대로 씁니다)
LEASE_SFT_MODEL_NAME = "muahseol/qwen3.5-0.8b-lease-sft"
lease_sft_model = AutoModelForCausalLM.from_pretrained(LEASE_SFT_MODEL_NAME, dtype=torch.float32).to(device)

### 7-2. 답 뽑고 선호 쌍 만들기

In [ ]:
# 범위 밖 질문마다 전월세 SFT 모델의 답을 4개씩 샘플링합니다(본실습과 같은 시드)
torch.manual_seed(0)
lease_train_samples = sample_replies(lease_sft_model, lease_train_refusal_questions, SYSTEM_PROMPT_LEASE)
lease_val_samples = sample_replies(lease_sft_model, lease_val_refusal_questions, SYSTEM_PROMPT_LEASE)

In [ ]:
# 🖐️ 함께 따라하기: 범위 밖 쌍(rejected 고르는 조건, chosen)과 전월세 질문 쌍(chosen, rejected)을 채우세요
lease_out_of_scope_pairs = {"train": [], "val": []}
for split, questions, samples in [("train", lease_train_refusal_questions, lease_train_samples), ("val", lease_val_refusal_questions, lease_val_samples)]:
    for question, answers in zip(questions, samples):
        # 거절 키워드가 없고 비어 있지 않은 첫 답이 rejected
        rejected = None
        for answer in answers:
            if ...:
                rejected = answer
                break
        if rejected is not None:
            lease_out_of_scope_pairs[split].append({"question": question, "chosen": ..., "rejected": rejected})

# 전월세 질문 쌍(학습은 범위 밖 학습 쌍의 절반): 학습 4행 간격 32행, 검증 8행 간격 4행. 과잉 거절을 막습니다
lease_pairs = {"train": [], "val": []}
for row in lease_train_rows[::4][:32]:
    lease_pairs["train"].append({"question": row["question"], "chosen": ..., "rejected": ...})
for row in lease_val_rows[::8][:4]:
    lease_pairs["val"].append({"question": row["question"], "chosen": ..., "rejected": ...})

# 대화형 Dataset 으로 바꿉니다(전월세 시스템 프롬프트)
lease_train_pairs = Dataset.from_list(
    [to_preference(pair["question"], pair["chosen"], pair["rejected"], SYSTEM_PROMPT_LEASE) for pair in lease_out_of_scope_pairs["train"] + lease_pairs["train"]]
)
lease_val_pairs = Dataset.from_list(
    [to_preference(pair["question"], pair["chosen"], pair["rejected"], SYSTEM_PROMPT_LEASE) for pair in lease_out_of_scope_pairs["val"] + lease_pairs["val"]]
)
print("범위 밖 쌍(학습, 검증):", len(lease_out_of_scope_pairs["train"]), len(lease_out_of_scope_pairs["val"]),
      "| 전월세 질문 쌍(학습, 검증):", len(lease_pairs["train"]), len(lease_pairs["val"]))

### 7-3. 학습

학습기를 만들 때 PEFT가 `base_model_name_or_path`가 노무 SFT 모델에서 전월세 SFT 모델로 바뀌었다는 경고를 냅니다. 본실습의 `lora_config`를 다시 써서 생기는 경고라 그대로 진행합니다.

In [ ]:
# 🖐️ 함께 따라하기: 본실습 beta 와 LoRA 설정을 채우세요
# dpo_config 와 같은 값입니다. 다른 줄에만 '다름'을 적었습니다
lease_dpo_config = DPOConfig(
    output_dir=str(output_dir / "lease_dpo_beta0.1"),  # 다름: 학습기가 기록을 둘 폴더(체크포인트는 저장하지 않습니다)
    beta=...,                                          # 본실습 첫 학습과 같은 beta
    loss_type=["sigmoid", "sft"],
    learning_rate=1e-4,                                # 다름: 본실습의 2배(거절 문장을 처음 배우는 모델)
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    seed=0,
    gradient_checkpointing=False,
    max_length=1024,
    eval_strategy="epoch",
    save_strategy="no",                                # 다름: 끝난 어댑터만 학습 뒤 save_model 로 저장합니다
    logging_steps=5,
    eval_on_start=True,
    report_to=["mlflow"],
    run_name="lease_dpo_beta_0.1",                     # 다름
)
lease_trainer = DPOTrainer(
    model=lease_sft_model,
    args=lease_dpo_config,
    train_dataset=lease_train_pairs,
    eval_dataset=lease_val_pairs,
    processing_class=tokenizer,
    peft_config=...,          # 어댑터를 끈 전월세 SFT 모델이 참조 모델이 됩니다
)

# 따라하기 run 은 본실습 실험과 나눠 따로 남깁니다
mlflow.set_experiment("lease-dpo")

# 쌍 수를 run 에 남기고 학습합니다(본실습과 같은 방법)
with mlflow.start_run(run_name=lease_dpo_config.run_name):
    mlflow.log_params({"sft_model": LEASE_SFT_MODEL_NAME, "out_of_scope_pairs": len(lease_out_of_scope_pairs["train"]),
                       "lease_pairs": len(lease_pairs["train"]), "val_pairs": len(lease_val_pairs)})
    lease_trainer.train()

### 7-4. 학습 기록과 정렬 전후 답

정렬 전 답은 LoRA 단원에서 배운 `disable_adapter()`로 어댑터를 끈 모델, 곧 참조 모델(전월세 SFT 모델)에서 얻습니다.

In [ ]:
# 에폭별 검증 손실과 보상 지표(맨 위는 학습 전)
lease_log_table = pd.DataFrame(lease_trainer.state.log_history)
lease_eval_log = lease_log_table.dropna(subset=["eval_loss"])
display(lease_eval_log[["epoch", "eval_loss", "eval_rewards/margins", "eval_rewards/accuracies"]].round(4))

In [ ]:
# 검증 범위 밖 질문 하나와 검증 전월세 질문 하나를 정렬 전(어댑터 끔)과 정렬 후에 묻습니다
lease_model = lease_trainer.model
lease_model.eval()
lease_questions = [lease_val_refusal_questions[0], lease_val_rows[0]["question"]]
with lease_model.disable_adapter():
    lease_before_answers = generate_replies(lease_model, lease_questions, SYSTEM_PROMPT_LEASE)
lease_after_answers = generate_replies(lease_model, lease_questions, SYSTEM_PROMPT_LEASE)
lease_check_table = pd.DataFrame({
    "질문": lease_questions,
    "SFT 모델 답변": lease_before_answers,
    "정렬 모델 답변": lease_after_answers,
})
display(lease_check_table)

**결과 읽기**
- **범위 밖 질문(주택임대사업자 세금):** 정렬 뒤에도 거절하지 않고 지어 답합니다. 검증 accuracies는 1에폭부터 0.8(20쌍 가운데 16쌍)에 머물렀지만, 이 질문은 아직 거절하지 않습니다.
- **전월세 질문(원룸 방 하나):** 정렬 뒤 답이 거절 문장의 첫머리('이 상담 챗봇이')로 시작하다가 중국어 문장으로 깨졌습니다. 거절 문장 쪽으로 옮긴 확률이 범위 안 질문에도 번진 모습입니다.
- 질문 두 개로는 판단할 수 없어, 교안 02 함께 따라하기에서 범위 밖 질문 10개로 거절률을, 전월세 평가셋 12문항으로 과잉 거절률을 계산합니다.

### 7-5. 저장

In [ ]:
# 학습한 어댑터를 저장합니다(교안 02 함께 따라하기가 전월세 SFT 모델에 얹어 평가합니다)
lease_trainer.save_model(output_dir / "lease_dpo_adapter")

## 8. 핵심 정리

- **DPO:** 선호 쌍에서 `chosen`($y_w$)의 확률을 `rejected`($y_l$)보다 높이는 학습입니다. 손실은 margin(`chosen`의 보상 - `rejected`의 보상)이 클수록 작아지고, 학습 첫 DPO 손실은 $\log 2 \approx 0.693$입니다. 손실이 차이만 보므로 `chosen`의 SFT 손실을 더했습니다(`loss_type=["sigmoid", "sft"]`).
- **참조 모델, 보상, beta:** 보상은 학습 중 모델이 참조 모델(어댑터를 끈 SFT 모델)보다 답의 로그 확률을 얼마나 바꿨는지에 beta를 곱한 값입니다. beta는 참조 모델과 달라지는 데 주는 벌의 무게입니다.
- **선호 쌍:** 범위 밖 쌍(`chosen` 거절 문장, `rejected` SFT 모델이 실제로 지어 낸 답)으로 거절을 가르치고, 과잉 거절을 막으려고 노무 질문 쌍(`chosen` 학습 데이터의 답, `rejected` 거절 문장)을 범위 밖 쌍의 절반만 넣었습니다.
- **학습:** 학습률 5e-5로 beta 0.1과 0.5를 3에폭씩 학습해 어댑터 6개를 저장했습니다. 학습 쌍의 margin은 크게 올랐지만 3에폭 끝 검증 margin / beta는 beta 0.1이 -0.09, beta 0.5가 -0.38로 0 근처였고, 확인 질문 가운데 발표 질문 하나가 거절로 바뀌었습니다.
- **기록과 답:** 학습 기록과 실제 답은 따로 움직일 수 있습니다. 모델이 새 질문을 거절하는지는 생성한 답으로 확인합니다.

교안 02에서는 이 어댑터 6개를 채점해 정렬 모델을 고르고, 퇴행을 다시 점검합니다.